### All days of the challange:

* [Day 1: Handling missing values](https://www.kaggle.com/rtatman/data-cleaning-challenge-handling-missing-values)
* [Day 2: Scaling and normalization](https://www.kaggle.com/rtatman/data-cleaning-challenge-scale-and-normalize-data)
* [Day 3: Parsing dates](https://www.kaggle.com/rtatman/data-cleaning-challenge-parsing-dates/)
* [Day 4: Character encodings](https://www.kaggle.com/rtatman/data-cleaning-challenge-character-encodings/)
* [Day 5: Inconsistent Data Entry](https://www.kaggle.com/rtatman/data-cleaning-challenge-inconsistent-data-entry/)
___

Welcome to day 5 of the 5-Day Data Challenge! (Can you believe it's already been five days??) Today, we're going to learn how to clean up inconsistent text entries. To get started, click the blue "Fork Notebook" button in the upper, right hand corner. This will create a private copy of this notebook that you can edit and play with. Once you're finished with the exercises, you can choose to make your notebook public to share with others. :)

> **Your turn!** As we work through this notebook, you'll see some notebook cells (a block of either code or text) that has "Your Turn!" written in it. These are exercises for you to do to help cement your understanding of the concepts we're talking about. Once you've written the code to answer a specific question, you can run the code by clicking inside the cell (box with code in it) with the code you want to run and then hit CTRL + ENTER (CMD + ENTER on a Mac). You can also click in a cell and then click on the right "play" arrow to the left of the code. If you want to run all the code in your notebook, you can use the double, "fast forward" arrows at the bottom of the notebook editor.

Here's what we're going to do today:

* [Get our environment set up](#Get-our-environment-set-up)
* [Do some preliminary text pre-processing](#Do-some-preliminary-text-pre-processing)
* [Use fuzzy matching to correct inconsistent data entry](#Use-fuzzy-matching-to-correct-inconsistent-data-entry)


Let's get started!

# Get our environment set up
________

The first thing we'll need to do is load in the libraries we'll be using. Not our datasets, though: we'll get to those later!

> **Important!** Make sure you run this cell yourself or the rest of your code won't work!

In [1]:
# modules we'll use
import pandas as pd
import numpy as np

# helpful modules
import fuzzywuzzy
from fuzzywuzzy import process
import chardet

# set seed for reproducibility
np.random.seed(0)

When I tried to read in the `PakistanSuicideAttacks Ver 11 (30-November-2017).csv`file the first time, I got a character encoding error, so I'm going to quickly check out what the encoding should be...

In [2]:
# look at the first ten thousand bytes to guess the character encoding
with open("../input/PakistanSuicideAttacks Ver 11 (30-November-2017).csv", 'rb') as rawdata:
    result = chardet.detect(rawdata.read(100000))

# check what the character encoding might be
print(result)

{'encoding': 'Windows-1252', 'confidence': 0.73, 'language': ''}


And then read it in with the correct encoding. (If this look unfamiliar to you, check out [yesterday's challenge](https://www.kaggle.com/rtatman/data-cleaning-challenge-character-encodings/).) 

In [3]:
# read in our dat
suicide_attacks = pd.read_csv("../input/PakistanSuicideAttacks Ver 11 (30-November-2017).csv", 
                              encoding='Windows-1252')

Now we're ready to get started! You can, as always, take a moment here to look at the data and get familiar with it. :)


# Do some preliminary text pre-processing
___

For this exercise, I'm interested in cleaning up the "City" column to make sure there's no data entry inconsistencies in it. We could go through and check each row by hand, of course, and hand-correct inconsistencies when we find them. There's a more efficient way to do this though!

In [4]:
# get all the unique values in the 'City' column
cities = suicide_attacks['City'].unique()

# sort them alphabetically and then take a closer look
cities.sort()
cities

array(['ATTOCK', 'Attock ', 'Bajaur Agency', 'Bannu', 'Bhakkar ', 'Buner',
       'Chakwal ', 'Chaman', 'Charsadda', 'Charsadda ', 'D. I Khan',
       'D.G Khan', 'D.G Khan ', 'D.I Khan', 'D.I Khan ', 'Dara Adam Khel',
       'Dara Adam khel', 'Fateh Jang', 'Ghallanai, Mohmand Agency ',
       'Gujrat', 'Hangu', 'Haripur', 'Hayatabad', 'Islamabad',
       'Islamabad ', 'Jacobabad', 'KURRAM AGENCY', 'Karachi', 'Karachi ',
       'Karak', 'Khanewal', 'Khuzdar', 'Khyber Agency', 'Khyber Agency ',
       'Kohat', 'Kohat ', 'Kuram Agency ', 'Lahore', 'Lahore ',
       'Lakki Marwat', 'Lakki marwat', 'Lasbela', 'Lower Dir', 'MULTAN',
       'Malakand ', 'Mansehra', 'Mardan', 'Mohmand Agency',
       'Mohmand Agency ', 'Mohmand agency', 'Mosal Kor, Mohmand Agency',
       'Multan', 'Muzaffarabad', 'North Waziristan', 'North waziristan',
       'Nowshehra', 'Orakzai Agency', 'Peshawar', 'Peshawar ', 'Pishin',
       'Poonch', 'Quetta', 'Quetta ', 'Rawalpindi', 'Sargodha',
       'Sehwan town',

Just looking at this, I can see some problems due to inconsistent data entry: 'Lahore' and 'Lahore ', for example, or 'Lakki Marwat' and 'Lakki marwat'.

The first thing I'm going to do is make everything lower case (I can change it back at the end if I like) and remove any white spaces at the beginning and end of cells. Inconsistencies in capitalizations and trailing white spaces are very common in text data and doing this often fixes many common text data entry inconsistencies.

In [5]:
# convert to lower case
suicide_attacks['City'] = suicide_attacks['City'].str.lower()
# remove trailing white spaces
suicide_attacks['City'] = suicide_attacks['City'].str.strip()

Next we're going to tackle more difficult inconsistencies.

In [6]:
# Your turn! Take a look at all the unique values in the "Province" column. 
# Then convert the column to lowercase and remove any trailing white spaces
def show_counts(series):
    # repr() rend visibles les espaces en début et fin de chaîne
    counts = series.value_counts(dropna=False)
    counts.index = [repr(value) for value in counts.index]
    print(counts.to_string())

print("Avant nettoyage :")
show_counts(suicide_attacks['Province'])

suicide_attacks['Province'] = suicide_attacks['Province'].str.lower().str.strip()

print("\nAprès nettoyage :")
show_counts(suicide_attacks['Province'])

Avant nettoyage :
'KPK'            251
'FATA'            70
'Punjab'          64
'Sindh'           35
'Baluchistan'     31
'Capital'         20
'Balochistan'     16
'AJK'              6
'Fata'             3

Après nettoyage :
'kpk'            251
'fata'            73
'punjab'          64
'sindh'           35
'baluchistan'     31
'capital'         20
'balochistan'     16
'ajk'              6


**Réponse :** avant nettoyage, la colonne compte 9 valeurs distinctes. La mise en minuscules fusionne `FATA` (70) et `Fata` (3) en `fata` (73), d'où 8 valeurs. Aucune valeur ne contenait d'espace parasite.

Il reste une incohérence que ce prétraitement ne peut pas corriger : **`baluchistan` (31 lignes) et `balochistan` (16 lignes)** sont deux translittérations du même nom ourdou. Elle est corrigée plus bas par fuzzy matching.

Pour lire les autres valeurs : `kpk` = Khyber Pakhtunkhwa, `fata` = *Federally Administered Tribal Areas* (zones tribales, intégrées à Khyber Pakhtunkhwa en 2018, après la date de ces données), `ajk` = Azad Jammu-et-Cachemire, `capital` = territoire de la capitale, Islamabad.

# Use fuzzy matching to correct inconsistent data entry
___

Alright, let's take another look at the city column and see if there's any more data cleaning we need to do.

In [7]:
# get all the unique values in the 'City' column
cities = suicide_attacks['City'].unique()

# sort them alphabetically and then take a closer look
cities.sort()
cities

array(['attock', 'bajaur agency', 'bannu', 'bhakkar', 'buner', 'chakwal',
       'chaman', 'charsadda', 'd. i khan', 'd.g khan', 'd.i khan',
       'dara adam khel', 'fateh jang', 'ghallanai, mohmand agency',
       'gujrat', 'hangu', 'haripur', 'hayatabad', 'islamabad',
       'jacobabad', 'karachi', 'karak', 'khanewal', 'khuzdar',
       'khyber agency', 'kohat', 'kuram agency', 'kurram agency',
       'lahore', 'lakki marwat', 'lasbela', 'lower dir', 'malakand',
       'mansehra', 'mardan', 'mohmand agency',
       'mosal kor, mohmand agency', 'multan', 'muzaffarabad',
       'north waziristan', 'nowshehra', 'orakzai agency', 'peshawar',
       'pishin', 'poonch', 'quetta', 'rawalpindi', 'sargodha',
       'sehwan town', 'shabqadar-charsadda', 'shangla', 'shikarpur',
       'sialkot', 'south waziristan', 'sudhanoti', 'sukkur', 'swabi',
       'swat', 'taftan', 'tangi, charsadda district', 'tank', 'taunsa',
       'tirah valley', 'totalai', 'upper dir', 'wagah', 'zhob'],
      dtype=

It does look like there are some remaining inconsistencies: 'd. i khan' and 'd.i khan' should probably be the same. (I [looked it up](https://en.wikipedia.org/wiki/List_of_most_populous_cities_in_Pakistan) and 'd.g khan' is a seperate city, so I shouldn't combine those.) 

I'm going to use the [fuzzywuzzy](https://github.com/seatgeek/fuzzywuzzy) package to help identify which string are closest to each other. This dataset is small enough that we could probably could correct errors by hand, but that approach doesn't scale well. (Would you want to correct a thousand errors by hand? What about ten thousand? Automating things as early as possible is generally a good idea. Plus, it’s fun! :)

> **Fuzzy matching:** The process of automatically finding text strings that are very similar to the target string. In general, a string is considered "closer" to another one the fewer characters you'd need to change if you were transforming one string into another. So "apple" and "snapple" are two changes away from each other (add "s" and "n") while "in" and "on" and one change away (rplace "i" with "o"). You won't always be able to rely on fuzzy matching 100%, but it will usually end up saving you at least a little time.

Fuzzywuzzy returns a ratio given two strings. The closer the ratio is to 100, the smaller the edit distance between the two strings. Here, we're going to get the ten strings from our list of cities that have the closest distance to "d.i khan".

In [8]:
# get the top 10 closest matches to "d.i khan"
matches = fuzzywuzzy.process.extract("d.i khan", cities, limit=10, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

# take a look at them
matches

[('d. i khan', 100),
 ('d.i khan', 100),
 ('d.g khan', 88),
 ('khanewal', 50),
 ('sudhanoti', 47),
 ('hangu', 46),
 ('kohat', 46),
 ('dara adam khel', 45),
 ('chaman', 43),
 ('mardan', 43)]

We can see that two of the items in the cities are very close to "d.i khan": "d. i khan" and "d.i khan". We can also see the "d.g khan", which is a seperate city, has a ratio of 88. Since we don't want to replace "d.g khan" with "d.i khan", let's replace all rows in our City column that have a ratio of at least 90 with "d.i khan". 

To do this, I'm going to write a function. (It's a good idea to write a general purpose function you can reuse if you think you might have to do a specific task more than once or twice. This keeps you from having to copy and paste code too often, which saves time and can help prevent mistakes.)

In [9]:
# function to replace rows in the provided column of the provided dataframe
# that match the provided string above the provided ratio with the provided string
def replace_matches_in_column(df, column, string_to_match, min_ratio=90):
    # get a list of unique non-missing strings
    strings = df[column].dropna().unique()

    # compare the input string with every distinct candidate
    matches = fuzzywuzzy.process.extract(
        string_to_match,
        strings,
        limit=len(strings),
        scorer=fuzzywuzzy.fuzz.token_sort_ratio
    )

    # only keep matches with a ratio >= min_ratio
    close_matches = [
        match for match, score in matches
        if score >= min_ratio
    ]

    # get the rows of all the close matches in our dataframe
    rows_with_matches = df[column].isin(close_matches)

    # replace all rows with close matches with the input match
    df.loc[rows_with_matches, column] = string_to_match

    # let us know the function's done
    print("All done!")

Now that we have a function, we can put it to the test!

In [10]:
# use the function we just wrote to replace close matches to "d.i khan" with "d.i khan"
replace_matches_in_column(df=suicide_attacks, column='City', string_to_match="d.i khan")

All done!


And now let's can check the unique values in our City column again and make sure we've tidied up d.i khan correctly.

In [11]:
# get all the unique values in the 'City' column
cities = suicide_attacks['City'].unique()

# sort them alphabetically and then take a closer look
cities.sort()
cities

array(['attock', 'bajaur agency', 'bannu', 'bhakkar', 'buner', 'chakwal',
       'chaman', 'charsadda', 'd.g khan', 'd.i khan', 'dara adam khel',
       'fateh jang', 'ghallanai, mohmand agency', 'gujrat', 'hangu',
       'haripur', 'hayatabad', 'islamabad', 'jacobabad', 'karachi',
       'karak', 'khanewal', 'khuzdar', 'khyber agency', 'kohat',
       'kuram agency', 'kurram agency', 'lahore', 'lakki marwat',
       'lasbela', 'lower dir', 'malakand', 'mansehra', 'mardan',
       'mohmand agency', 'mosal kor, mohmand agency', 'multan',
       'muzaffarabad', 'north waziristan', 'nowshehra', 'orakzai agency',
       'peshawar', 'pishin', 'poonch', 'quetta', 'rawalpindi', 'sargodha',
       'sehwan town', 'shabqadar-charsadda', 'shangla', 'shikarpur',
       'sialkot', 'south waziristan', 'sudhanoti', 'sukkur', 'swabi',
       'swat', 'taftan', 'tangi, charsadda district', 'tank', 'taunsa',
       'tirah valley', 'totalai', 'upper dir', 'wagah', 'zhob'],
      dtype=object)

Excellent! Now we only have "d.i khan" in our dataframe and we didn't have to change anything by hand. 

In [12]:
# Your turn! It looks like 'kuram agency' and 'kurram agency' should
# be the same city. Correct the dataframe so that they are.

# 1) On examine d'abord les scores, pour vérifier qu'aucune autre ville ne dépasse le seuil par erreur
cities = suicide_attacks['City'].unique()
print("Correspondances les plus proches de « kurram agency » :")
print(fuzzywuzzy.process.extract("kurram agency", cities, limit=5, scorer=fuzzywuzzy.fuzz.token_sort_ratio))

# 2) Remplacement par l'orthographe correcte (Kurram Agency, dans les zones tribales)
n_before = suicide_attacks['City'].isin(['kuram agency', 'kurram agency']).sum()
replace_matches_in_column(df=suicide_attacks, column='City', string_to_match="kurram agency")

# 3) Vérification
remaining = [city for city in suicide_attacks['City'].unique() if 'kur' in str(city)]
print(f"\nValeurs restantes contenant « kur » : {remaining}")
print(f"Lignes concernées : {n_before} avant ; {(suicide_attacks['City'] == 'kurram agency').sum()} « kurram agency » après")

Correspondances les plus proches de « kurram agency » :
[('kurram agency', 100), ('kuram agency', 96), ('bajaur agency', 69), ('khyber agency', 69), ('orakzai agency', 67)]
All done!

Valeurs restantes contenant « kur » : ['kurram agency', 'sukkur']
Lignes concernées : 6 avant ; 6 « kurram agency » après


**Réponse :** `kuram agency` obtient un score de **96**, au-dessus du seuil de 90. Les candidats suivants (`bajaur agency` et `khyber agency`, à 69) en sont très loin : cet **écart entre 96 et 69** montre qu'avec les valeurs présentes ici, le seuil de 90 ne fusionne pas ces autres villes. Les 6 lignes concernées portent désormais l'orthographe retenue, `kurram agency`.

**Les mathématiques derrière le score.**

*Distance de Levenshtein.* C'est le nombre minimal d'insertions, de suppressions et de substitutions pour passer d'une chaîne $a$ à une chaîne $b$. Elle se calcule par programmation dynamique, en notant $d(i,j)$ la distance entre les $i$ premiers caractères de $a$ et les $j$ premiers de $b$ :
$$d(i,j)=\min\left(d(i-1,j)+1,\;d(i,j-1)+1,\;d(i-1,j-1)+\mathbb{1}[a_i\neq b_j]\right),$$
avec $d(i,0)=i$ et $d(0,j)=j$. Ici, une seule insertion, le second « r », suffit : $d=1$.

*Score de `fuzzywuzzy`.* `fuzz.ratio` utilise un objet `SequenceMatcher`. Selon l'installation, FuzzyWuzzy peut utiliser `python-Levenshtein` ou son repli en Python pur fondé sur `difflib`. Il ne faut donc pas attribuer inconditionnellement le calcul à un seul de ces moteurs. Pour `kuram agency` et `kurram agency`, le score affiché dans ce notebook est **96**.

*Variante `token_sort_ratio`.* Avant de calculer le score, elle met le texte en minuscules, **remplace la ponctuation par des espaces**, normalise les espaces et **trie les mots**. C'est pourquoi `d. i khan` et `d.i khan` deviennent tous deux `d i khan` et obtiennent 100. Inversement, `d.i khan` et `d.g khan` restent très proches graphiquement et obtiennent 88, alors qu'ils désignent des lieux différents.

In [13]:
# Bonus (« More practice ») : au lieu de deviner les doublons un par un,
# on compare TOUTES les paires de villes distinctes et on liste les plus proches.
from itertools import combinations
from fuzzywuzzy import fuzz

city_counts = suicide_attacks['City'].value_counts()
unique_cities = list(city_counts.index)
n_pairs = len(unique_cities) * (len(unique_cities) - 1) // 2

candidates = sorted(
    ((fuzz.token_sort_ratio(a, b), a, b) for a, b in combinations(unique_cities, 2)),
    reverse=True)

print(f"{len(unique_cities)} villes distinctes, {n_pairs} paires comparées\n")
print("Paires les plus proches (score ≥ 80), à valider à la main :")
for score, a, b in candidates:
    if score < 80:
        break
    print(f"{score:>3}  {a!r} ({city_counts[a]} lignes)  ~  {b!r} ({city_counts[b]} lignes)")

65 villes distinctes, 2080 paires comparées

Paires les plus proches (score ≥ 80), à valider à la main :
 88  'north waziristan' (24 lignes)  ~  'south waziristan' (11 lignes)
 88  'd.i khan' (13 lignes)  ~  'd.g khan' (4 lignes)


**Analyse.** On compare les $\binom{65}{2}=2\,080$ paires de villes distinctes. Après nettoyage, seules deux paires dépassent un score de 80, et ce sont **deux faux positifs** :
- `north waziristan` et `south waziristan` (88) : deux régions distinctes ;
- `d.i khan` et `d.g khan` (88) : Dera Ismail Khan et Dera Ghazi Khan, deux villes différentes.

**Leçon : le score mesure une ressemblance de caractères, pas de sens.** Le vrai doublon `kuram agency` / `kurram agency` obtenait 96 ; ces lieux distincts obtiennent 88. Sur des chaînes courtes, une seule lettre peut changer le lieu désigné. Le choix du seuil est un compromis : trop bas, on fusionne des lieux différents (faux positifs) ; trop haut, on manque de vrais doublons (faux négatifs). C'est pourquoi les candidats doivent être **validés à la main** ou confrontés à une liste de référence.

**Une incohérence que le fuzzy matching ne voit pas : le niveau géographique.** Certaines valeurs désignent un quartier ou une localité plutôt qu'une ville : `hayatabad` (quartier de Peshawar), `mosal kor, mohmand agency` et `ghallanai, mohmand agency` (localités de Mohmand Agency), `tangi, charsadda district` et `shabqadar-charsadda` (district de Charsadda), `tirah valley` (vallée s'étendant entre les régions de Khyber et d'Orakzai). Leurs noms sont très différents, donc aucun score ne les rapproche. Les harmoniser suppose une table de référence géographique et un choix explicite du niveau d'analyse.

**Complexité.** Comparer toutes les paires coûte $O(n^2)$ : 2 080 comparaisons ici, mais environ 50 millions pour 10 000 valeurs distinctes. Sur de gros volumes, on ne compare que les valeurs d'un même « bloc », par exemple celles qui partagent la même première lettre (*blocking*).

In [14]:
# More practice : d'autres colonnes ont-elles des saisies incohérentes ?

# 1) Province : « baluchistan » et « balochistan » sont deux orthographes du même nom.
#    On vérifie les scores, puis on garde l'orthographe officielle actuelle, « balochistan ».
provinces = suicide_attacks['Province'].unique()
print(fuzzywuzzy.process.extract("balochistan", provinces, limit=3, scorer=fuzzywuzzy.fuzz.token_sort_ratio))
replace_matches_in_column(df=suicide_attacks, column='Province', string_to_match="balochistan")
show_counts(suicide_attacks['Province'])

# 2) Balayage de toutes les colonnes texte : combien de valeurs distinctes disparaissent
#    rien qu'en normalisant la casse et les espaces ?
print("\nColonnes où la casse ou les espaces créent des doublons :")
for col in suicide_attacks.select_dtypes(include='object').columns:
    values = suicide_attacks[col].dropna().astype(str)
    normalized = values.str.lower().str.strip().str.replace(r'\s+', ' ')
    n_raw, n_norm = values.nunique(), normalized.nunique()
    if n_norm < n_raw:
        print(f"   {col!r} : {n_raw} valeurs distinctes → {n_norm} après normalisation")

[('balochistan', 100), ('baluchistan', 91), ('capital', 44)]
All done!
'kpk'            251
'fata'            73
'punjab'          64
'balochistan'     47
'sindh'           35
'capital'         20
'ajk'              6

Colonnes où la casse ou les espaces créent des doublons :
   'Time' : 215 valeurs distinctes → 212 après normalisation
   'Location' : 487 valeurs distinctes → 485 après normalisation
   'Location Category' : 25 valeurs distinctes → 24 après normalisation
   'Location Sensitivity' : 4 valeurs distinctes → 3 après normalisation
   'Open/Closed Space' : 6 valeurs distinctes → 3 après normalisation
   'Influencing Event/Event' : 173 valeurs distinctes → 168 après normalisation
   'Target Type' : 24 valeurs distinctes → 19 après normalisation
   'Targeted Sect if any' : 8 valeurs distinctes → 7 après normalisation
   'Explosive Weight (max)' : 114 valeurs distinctes → 81 après normalisation
   'Hospital Names' : 239 valeurs distinctes → 227 après normalisation


In [15]:
# 3) Application : on normalise les colonnes catégorielles concernées (peu de valeurs distinctes)
categorical_cols = ['Location Category', 'Location Sensitivity', 'Open/Closed Space',
                    'Target Type', 'Targeted Sect if any']
for col in categorical_cols:
    n_before = suicide_attacks[col].nunique()
    suicide_attacks[col] = suicide_attacks[col].str.lower().str.strip().str.replace(r'\s+', ' ')
    print(f"{col!r} : {n_before} → {suicide_attacks[col].nunique()} valeurs distinctes")

print("\nValeurs de 'Open/Closed Space' après nettoyage :")
show_counts(suicide_attacks['Open/Closed Space'])

# 4) « Explosive Weight (max) » devrait être numérique : un échantillon montre pourquoi
#    une simple normalisation du texte ne suffit pas
print("\nÉchantillon de 'Explosive Weight (max)' :")
print(suicide_attacks['Explosive Weight (max)'].dropna().drop_duplicates().sample(10, random_state=0).tolist())

'Location Category' : 25 → 24 valeurs distinctes
'Location Sensitivity' : 4 → 3 valeurs distinctes
'Open/Closed Space' : 6 → 3 valeurs distinctes
'Target Type' : 24 → 19 valeurs distinctes
'Targeted Sect if any' : 8 → 7 valeurs distinctes

Valeurs de 'Open/Closed Space' après nettoyage :
'open'           333
'closed'         127
nan               35
'open/closed'      1

Échantillon de 'Explosive Weight (max)' :
['24Kg', '15 kg', '4kg-5kg', '10 to 15 kg', ' 10 kg ', '250kg', '10KG', '40 kg', '1000kg', '2kg']


**Réponse (More practice) : d'autres colonnes ont-elles des saisies incohérentes ?** Oui, plusieurs.

**1. `Province`.** `baluchistan` et `balochistan` ne diffèrent que d'une substitution (« u » / « o ») et obtiennent un score de 91, au-dessus du seuil de 90. Après fusion, on obtient **7 modalités administratives** dans cette colonne, dont `balochistan` (47 lignes). Le terme « provinces » serait impropre ici, car la colonne contient aussi des territoires ou catégories administratives qui ne sont pas des provinces au sens strict.

**2. Balayage de toutes les colonnes texte.** Une simple normalisation (minuscules, suppression des espaces aux extrémités, espaces multiples réduits à un seul) réduit le nombre de valeurs distinctes dans **10 colonnes**. Pour `City`, elle explique 26 des 28 fusions (93 → 67, contre 67 → 65 par fuzzy matching), ce qui va dans le sens de l'affirmation du notebook : casse et espaces expliquent l'essentiel des incohérences de saisie.

**3. Application aux colonnes catégorielles** (peu de modalités, où chaque doublon fausse les comptages) :

| Colonne | Avant | Après |
|---|---|---|
| `Location Category` | 25 | 24 |
| `Location Sensitivity` | 4 | 3 |
| `Open/Closed Space` | 6 | 3 |
| `Target Type` | 24 | 19 |
| `Targeted Sect if any` | 8 | 7 |

Sans ce nettoyage, un `value_counts()` sur `Open/Closed Space` aurait éclaté les mêmes catégories en **deux fois plus de modalités**. Après nettoyage : `open` (333 lignes), `closed` (127), 35 valeurs manquantes, et une valeur ambiguë, `open/closed` (1 ligne), qu'il faudrait vérifier à la source.

**4. Les colonnes laissées de côté, et pourquoi.**
- **Texte libre** (`Location`, `Hospital Names`, `Influencing Event/Event`) : ces colonnes comportent beaucoup de valeurs distinctes. On les dédoublonnerait comme les villes, par fuzzy matching suivi d'une validation manuelle.
- **`Time`** (215 → 212) : il s'agit surtout d'un problème de **type** (3 doublons de casse ou d'espaces seulement) : il faut convertir en heure, comme pour les dates au notebook 3.
- **`Explosive Weight (max)`** (114 → 81 valeurs) : ce sont des **nombres stockés comme du texte**. L'échantillon montre trois difficultés : l'unité écrite de plusieurs façons (`24Kg`, `10KG`, `15 kg`), des espaces parasites (`' 10 kg '`) et surtout des **intervalles** (`4kg-5kg`, `10 to 15 kg`). Pour la rendre numérique, il faudrait extraire les nombres par expression régulière, puis décider comment traiter un intervalle. Comme la colonne s'appelle « (max) », la borne supérieure est le choix cohérent.

## Synthèse

| Étape | Constat | Choix et justification |
|---|---|---|
| `City` : casse et espaces | 93 valeurs brutes → 67 après minuscules et suppression des espaces | Corrige la majorité des incohérences en une ligne |
| `City` : fautes de frappe | `d. i khan` / `d.i khan` (100) ; `kuram agency` / `kurram agency` (96) | Fuzzy matching avec un seuil de 90 → 65 villes |
| Détection systématique | 2 080 paires comparées ; les deux seules au-dessus de 80 sont de faux positifs (88) | Validation humaine indispensable : le score mesure les caractères, pas le sens |
| `Province` | `FATA` / `Fata` (casse) ; `baluchistan` / `balochistan` (orthographe, 91) | Normalisation puis fuzzy matching → 7 modalités administratives |
| Autres colonnes | 10 colonnes avec des doublons de casse ou d'espaces | 5 colonnes catégorielles nettoyées ; `Time` et `Explosive Weight (max)` relèvent d'une conversion de type |

**À retenir :** le nettoyage de texte se fait par couches. La normalisation (casse, espaces) traite la majorité des cas, le fuzzy matching les fautes de frappe, et la validation humaine décide de ce qui désigne vraiment la même chose.

And that's it for today! If you have any questions, be sure to post them in the comments below or [on the forums](https://www.kaggle.com/questions-and-answers). 

Remember that your notebook is private by default, and in order to share it with other people or ask for help with it, you'll need to make it public. First, you'll need to save a version of your notebook that shows your current work by hitting the "Commit & Run" button. (Your work is saved automatically, but versioning your work lets you go back and look at what it was like at the point you saved it. It also lets you share a nice compiled notebook instead of just the raw code.) Then, once your notebook is finished running, you can go to the Settings tab in the panel to the left (you may have to expand it by hitting the [<] button next to the "Commit & Run" button) and setting the "Visibility" dropdown to "Public".

# More practice!
___

Do any other columns in this dataframe have inconsistent data entry? If you can find any, try to tidy them up.

You can also try reading in the `PakistanSuicideAttacks Ver 6 (10-October-2017).csv` file from this dataset and tidying up any inconsistent columns in that data file.